# Nutrition5k mass estimation: pseudo-depth v3 with semantic segmentation

Uses the semantic food segmentation model `models/yolo_food_sem.pt`


In [ ]:
from pathlib import Path
import json
import math
import os
import sys
import time
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor, Pool
from ultralytics import YOLO


## 2. Project paths and settings

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if all((path / name).exists() for name in ("DepthModule", "mass_estimation", "nutrition5k", "models")):
            return path
    raise RuntimeError("Project root was not found. Run the notebook from inside the cloned repository.")


def env_path(name: str, default: Path) -> Path:
    return Path(os.getenv(name, default)).expanduser()


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "DepthModule"))
from depth_module import DepthAnything3Module

MASS_DIR = PROJECT_ROOT / "mass_estimation"
RESULTS_DIR = MASS_DIR / "with_semantic"
MAPPING_DIR = PROJECT_ROOT / "class_mappings"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NUTRITION_CSV = env_path("NUTRITION_CSV", PROJECT_ROOT / "nutrition5k" / "dish_nutrition_values.csv")
OVERHEAD_DIR = env_path("OVERHEAD_DIR", PROJECT_ROOT / "nutrition5k" / "imagery" / "realsense_overhead")
SEG_MODEL_PATH = env_path("SEG_MODEL_PATH", PROJECT_ROOT / "models" / "yolo_food_sem.pt")
PLATE_MODEL_PATH = env_path("PLATE_MODEL_PATH", PROJECT_ROOT / "models" / "plate_seg.pt")

_cls_default = PROJECT_ROOT / "models" / "yolo_cls.pt"
_cls_override = os.getenv("CLS_MODEL_PATH")
CLS_MODEL_PATH = Path(_cls_override).expanduser() if _cls_override else (_cls_default if _cls_default.exists() else None)
DEPTH_MODEL_ID = os.getenv("DEPTH_MODEL", "depth-anything-v3-base")

FEATURES_CSV = RESULTS_DIR / "catboost_pseudodepth_v3_with_semantic_features.csv"
MODEL_PATH = RESULTS_DIR / "catboost_pseudodepth_v3_with_semantic.cbm"
METRICS_PATH = RESULTS_DIR / "catboost_pseudodepth_v3_with_semantic_metrics.json"
VALID_REPORT_CSV = RESULTS_DIR / "catboost_pseudodepth_v3_with_semantic_valid_report.csv"

RANDOM_STATE = 42
MAX_DISHES = None
RESUME_FEATURE_CACHE = True
YOLO_CONF = 0.25
YOLO_IMGSZ = 640
CACHE_EVERY_N_ROWS = 25

CATBOOST_PARAMS = dict(
    loss_function="MAE",
    eval_metric="MAE",
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=8,
    random_seed=RANDOM_STATE,
    od_type="Iter",
    od_wait=150,
    verbose=100,
)

required_paths = [NUTRITION_CSV, OVERHEAD_DIR, SEG_MODEL_PATH, PLATE_MODEL_PATH, MAPPING_DIR]
missing = [str(p.relative_to(PROJECT_ROOT) if p.is_relative_to(PROJECT_ROOT) else p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required project paths: " + ", ".join(missing))

print("Project root:", PROJECT_ROOT)
print("Nutrition CSV:", NUTRITION_CSV.relative_to(PROJECT_ROOT))
print("Overhead images:", OVERHEAD_DIR.relative_to(PROJECT_ROOT))
print("Food semantic segmentation model:", SEG_MODEL_PATH.relative_to(PROJECT_ROOT))
print("Plate segmentation model:", PLATE_MODEL_PATH.relative_to(PROJECT_ROOT))
print("Classification model:", CLS_MODEL_PATH.relative_to(PROJECT_ROOT) if CLS_MODEL_PATH else "disabled")
print("Depth model:", DEPTH_MODEL_ID)
print("Outputs:", RESULTS_DIR.relative_to(PROJECT_ROOT))


## 3. Metadata


In [ ]:
seg_map = pd.read_csv(MAPPING_DIR / "foodseg103_density_groups.csv")
seg_by_id = seg_map.set_index("class_id").to_dict("index")
seg_by_name = {str(row["class_name"]).strip().casefold(): row for row in seg_map.to_dict("records")}
food101_map = pd.read_csv(MAPPING_DIR / "food101_dish_groups.csv")
food101_group_by_name = dict(zip(food101_map["class_name"].astype(str), food101_map["dish_group"].astype(str)))

density_groups = sorted(seg_map.loc[seg_map["use_for_mask"].astype(str).str.lower() == "true", "density_group"].unique())
nutrition_cols = ["calories", "fat", "carb", "protein"]
labels = (
    pd.read_csv(NUTRITION_CSV)[["dish_id", "mass", *nutrition_cols]]
    .dropna(subset=["dish_id", "mass"])
    .assign(rgb_path=lambda df: df["dish_id"].astype(str).map(lambda dish_id: OVERHEAD_DIR / dish_id / "rgb.png"))
)
labels = labels[labels["rgb_path"].map(Path.exists)].copy()
if MAX_DISHES is not None:
    labels = labels.head(MAX_DISHES).copy()

print("Density groups:", density_groups)
print("Food101 groups:", sorted(food101_map["dish_group"].unique()))
print(f"Rows with rgb.png and mass target: {len(labels):,}")
labels.head()


## 4. Feature engineering helpers


In [ ]:
def resize_nearest(mask: np.ndarray, shape_hw: Tuple[int, int]) -> np.ndarray:
    if mask.shape[:2] == shape_hw:
        return mask.astype(bool)
    return cv2.resize(mask.astype(np.uint8), (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_NEAREST).astype(bool)

def resize_depth(depth: np.ndarray, shape_hw: Tuple[int, int]) -> np.ndarray:
    depth = np.asarray(depth, dtype=np.float32)
    if depth.shape[:2] == shape_hw:
        return depth
    return cv2.resize(depth, (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_LINEAR)

def normalize_depth(depth: np.ndarray) -> Dict[str, np.ndarray]:
    depth = np.asarray(depth, dtype=np.float32)
    finite = depth[np.isfinite(depth)]
    if finite.size == 0:
        z = np.zeros_like(depth, dtype=np.float32)
        return {"raw": z, "p05p95": z, "iqr_z": z}
    p05, p25, p50, p75, p95 = np.percentile(finite, [5, 25, 50, 75, 95])
    scale = max(float(p95 - p05), 1e-6)
    iqr = max(float(p75 - p25), 1e-6)
    p05p95 = np.clip((depth - p05) / scale, 0.0, 1.0).astype(np.float32)
    iqr_z = np.clip((depth - p50) / iqr, -5.0, 5.0).astype(np.float32)
    return {"raw": depth, "p05p95": p05p95, "iqr_z": iqr_z}

def mask_shape_features(mask: np.ndarray, prefix: str) -> Dict[str, float]:
    mask_u8 = mask.astype(np.uint8)
    area = int(mask_u8.sum())
    out = {f"{prefix}_area_px": area}
    if area == 0:
        out.update({
            f"{prefix}_perimeter": 0.0,
            f"{prefix}_compactness": 0.0,
            f"{prefix}_solidity": 0.0,
            f"{prefix}_equiv_diameter": 0.0,
        })
        return out
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = float(sum(cv2.arcLength(c, True) for c in contours))
    hull_area = 0.0
    for c in contours:
        if len(c) >= 3:
            hull_area += float(cv2.contourArea(cv2.convexHull(c)))
    out.update({
        f"{prefix}_perimeter": perimeter,
        f"{prefix}_compactness": float(area / max(perimeter * perimeter, 1e-6)),
        f"{prefix}_solidity": float(area / max(hull_area, 1e-6)),
        f"{prefix}_equiv_diameter": float(math.sqrt(4.0 * area / math.pi)),
    })
    return out


In [ ]:
def semantic_class_metadata(seg_model: YOLO, model_class_id: int) -> Tuple[str, Dict[str, object]]:
    model_class_name = str(seg_model.names.get(int(model_class_id), model_class_id))
    meta = seg_by_name.get(model_class_name.strip().casefold())
    if meta is None:
        meta = seg_by_id.get(int(model_class_id), {})
    class_name = str(meta.get("class_name", model_class_name)) if meta else model_class_name
    return class_name, meta


def predict_semantic_segments(seg_model: YOLO, image: Image.Image) -> Tuple[List[Dict[str, object]], Dict[str, float]]:
    image_np = np.array(image.convert("RGB"))
    h, w = image_np.shape[:2]
    result = seg_model.predict(image_np, imgsz=YOLO_IMGSZ, verbose=False)[0]
    semantic_mask = getattr(result, "semantic_mask", None)
    if semantic_mask is None or getattr(semantic_mask, "data", None) is None:
        return [], {
            "n_masks_raw": 0,
            "n_masks_kept": 0,
            "n_semantic_classes_raw": 0,
            "n_semantic_classes_kept": 0,
            "seg_conf_mean": 0.0,
            "seg_conf_max": 0.0,
        }

    semantic_map = semantic_mask.data.detach().cpu().numpy().astype(np.int32)
    if semantic_map.shape[:2] != (h, w):
        semantic_map = cv2.resize(semantic_map.astype(np.uint16), (w, h), interpolation=cv2.INTER_NEAREST).astype(np.int32)

    class_ids, counts = np.unique(semantic_map, return_counts=True)
    segments = []
    for model_class_id, area in zip(class_ids.tolist(), counts.tolist()):
        class_name, meta = semantic_class_metadata(seg_model, int(model_class_id))
        use_for_mask = str(meta.get("use_for_mask", "true")).strip().lower() == "true" if meta else True
        if not use_for_mask:
            continue
        mask = semantic_map == int(model_class_id)
        area = int(area)
        if area == 0:
            continue
        segments.append({
            "mask": mask,
            "class_id": int(meta.get("class_id", model_class_id)) if meta else int(model_class_id),
            "model_class_id": int(model_class_id),
            "class_name": class_name,
            "density_group": str(meta.get("density_group", "unknown")) if meta else "unknown",
            "conf": 0.0,
            "area": area,
        })

    stats = {
        "n_masks_raw": int(len(class_ids)),
        "n_masks_kept": int(len(segments)),
        "n_semantic_classes_raw": int(len(class_ids)),
        "n_semantic_classes_kept": int(len(segments)),
        # The semantic model returns a hard class map, not confidence scores.
        "seg_conf_mean": 0.0,
        "seg_conf_max": 0.0,
    }
    return segments, stats


def predict_plate_mask(plate_model: YOLO, image: Image.Image) -> Tuple[np.ndarray, Dict[str, float]]:
    image_np = np.array(image.convert("RGB"))
    h, w = image_np.shape[:2]
    result = plate_model.predict(image_np, retina_masks=True, conf=YOLO_CONF, imgsz=YOLO_IMGSZ, verbose=False)[0]
    empty = np.zeros((h, w), dtype=bool)
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return empty, {"n_plate_masks": 0, "plate_conf_mean": 0.0, "plate_conf_max": 0.0}

    raw_masks = result.masks.data.cpu().numpy() > 0.5
    confs = result.boxes.conf.cpu().numpy().astype(float)
    masks = [resize_nearest(raw_mask, (h, w)) for raw_mask in raw_masks]
    masks = [mask for mask in masks if int(mask.sum()) > 0]
    if not masks:
        return empty, {"n_plate_masks": int(len(raw_masks)), "plate_conf_mean": float(np.mean(confs)), "plate_conf_max": float(np.max(confs))}

    plate_mask = np.logical_or.reduce(masks)
    stats = {
        "n_plate_masks": int(len(raw_masks)),
        "plate_conf_mean": float(np.mean(confs)) if len(confs) else 0.0,
        "plate_conf_max": float(np.max(confs)) if len(confs) else 0.0,
    }
    return plate_mask, stats


def predict_food101(cls_model: Optional[YOLO], image: Image.Image, topk: int = 5) -> Dict[str, object]:
    out = {"food101_top1": "unknown", "food101_top1_conf": 0.0, "food101_dish_group": "unknown", "food101_entropy": 0.0}
    for k in range(2, topk + 1):
        out[f"food101_top{k}"] = "unknown"
        out[f"food101_top{k}_conf"] = 0.0
    if cls_model is None:
        return out
    result = cls_model.predict(np.array(image.convert("RGB")), imgsz=YOLO_IMGSZ, verbose=False)[0]
    probs = getattr(result, "probs", None)
    if probs is None:
        return out
    top_ids = list(probs.top5[:topk])
    top_confs = [float(x) for x in probs.top5conf[:topk]]
    for i, (class_id, conf) in enumerate(zip(top_ids, top_confs), start=1):
        name = str(cls_model.names.get(int(class_id), class_id))
        out[f"food101_top{i}"] = name
        out[f"food101_top{i}_conf"] = conf
        if i == 1:
            out["food101_dish_group"] = food101_group_by_name.get(name, "unknown")
    conf_arr = np.asarray(top_confs, dtype=np.float32)
    conf_arr = conf_arr / max(float(conf_arr.sum()), 1e-6)
    out["food101_entropy"] = float(-(conf_arr * np.log(conf_arr + 1e-9)).sum())
    return out


In [ ]:
def height_features(mask: np.ndarray, depth_norm: np.ndarray, prefix: str, ring_fracs=(0.015, 0.035, 0.07)) -> Dict[str, float]:
    h, w = mask.shape[:2]
    out = {}
    food_values = depth_norm[mask]
    food_values = food_values[np.isfinite(food_values)]
    if food_values.size == 0:
        for frac in ring_fracs:
            tag = f"{prefix}_ring{int(frac * 1000):03d}"
            out[f"{tag}_plate_depth"] = 0.0
            out[f"{tag}_vol_plate_minus_food"] = 0.0
            out[f"{tag}_vol_food_minus_plate"] = 0.0
            out[f"{tag}_mean_abs_height"] = 0.0
            out[f"{tag}_p75_abs_height"] = 0.0
            out[f"{tag}_p95_abs_height"] = 0.0
        return out

    for frac in ring_fracs:
        kernel_size = max(5, int(round(min(h, w) * frac)))
        if kernel_size % 2 == 0:
            kernel_size += 1
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        dilated = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1).astype(bool)
        ring = dilated & ~mask
        ring_values = depth_norm[ring]
        ring_values = ring_values[np.isfinite(ring_values)]
        if ring_values.size < 20:
            ring_values = depth_norm[~mask]
            ring_values = ring_values[np.isfinite(ring_values)]
        plate_depth = float(np.median(ring_values)) if ring_values.size else float(np.median(depth_norm[np.isfinite(depth_norm)]))
        plate_minus_food = np.clip(plate_depth - food_values, 0, None)
        food_minus_plate = np.clip(food_values - plate_depth, 0, None)
        if plate_minus_food.size:
            plate_minus_food = np.clip(plate_minus_food, 0, np.percentile(plate_minus_food, 95))
        if food_minus_plate.size:
            food_minus_plate = np.clip(food_minus_plate, 0, np.percentile(food_minus_plate, 95))
        tag = f"{prefix}_ring{int(frac * 1000):03d}"
        out[f"{tag}_plate_depth"] = plate_depth
        out[f"{tag}_vol_plate_minus_food"] = float(plate_minus_food.sum())
        out[f"{tag}_vol_food_minus_plate"] = float(food_minus_plate.sum())
        out[f"{tag}_mean_abs_height"] = float(np.mean(np.abs(food_values - plate_depth)))
        out[f"{tag}_p75_abs_height"] = float(np.percentile(np.abs(food_values - plate_depth), 75))
        out[f"{tag}_p95_abs_height"] = float(np.percentile(np.abs(food_values - plate_depth), 95))
    return out


def plate_relation_features(food_mask: np.ndarray, plate_mask: np.ndarray, image_area: int) -> Dict[str, float]:
    food_area = int(food_mask.sum())
    plate_area = int(plate_mask.sum())
    intersection = int((food_mask & plate_mask).sum()) if plate_area else 0
    return {
        "plate_area_px": plate_area,
        "plate_area_ratio": float(plate_area / max(image_area, 1)),
        "food_to_plate_area_ratio": float(food_area / max(plate_area, 1)) if plate_area else 0.0,
        "food_plate_intersection_px": intersection,
        "food_in_plate_ratio": float(intersection / max(food_area, 1)) if food_area else 0.0,
        "plate_covered_by_food_ratio": float(intersection / max(plate_area, 1)) if plate_area else 0.0,
    }


def compute_image_features(segments: List[Dict[str, object]], plate_mask: np.ndarray, depth: np.ndarray, image_size: Tuple[int, int]) -> Dict[str, object]:
    h, w = image_size
    depth = resize_depth(depth, (h, w))
    depth_versions = normalize_depth(depth)
    plate_mask = resize_nearest(plate_mask, (h, w))

    out: Dict[str, object] = {"image_h": h, "image_w": w, "image_area_px": h * w}
    for group in density_groups:
        out[f"area_group_{group}"] = 0.0
        out[f"count_group_{group}"] = 0.0
        out[f"pvol_group_{group}"] = 0.0

    if not segments:
        empty_food = np.zeros((h, w), dtype=bool)
        out.update({
            "area_px": 0,
            "sqrt_area": 0.0,
            "log_area": 0.0,
            "area_ratio": 0.0,
            "n_unique_seg_classes": 0,
            "dominant_density_group": "unknown",
            "dominant_seg_class": "unknown",
        })
        for rank in range(1, 4):
            out[f"top{rank}_seg_class"] = "unknown"
            out[f"top{rank}_density_group"] = "unknown"
            out[f"top{rank}_area_ratio"] = 0.0
        out.update(mask_shape_features(empty_food, "union"))
        out.update(mask_shape_features(plate_mask, "plate"))
        out.update(plate_relation_features(empty_food, plate_mask, h * w))
        return out

    union_mask = np.logical_or.reduce([segment["mask"] for segment in segments])
    area_px = int(union_mask.sum())
    out.update({
        "area_px": area_px,
        "sqrt_area": float(math.sqrt(area_px)),
        "log_area": float(math.log1p(area_px)),
        "area_ratio": float(area_px / max(h * w, 1)),
        "n_unique_seg_classes": int(len(set(segment["class_id"] for segment in segments))),
    })
    out.update(mask_shape_features(union_mask, "union"))
    out.update(mask_shape_features(plate_mask, "plate"))
    out.update(plate_relation_features(union_mask, plate_mask, h * w))

    segments_sorted = sorted(segments, key=lambda x: int(x["area"]), reverse=True)
    for rank in range(1, 4):
        if len(segments_sorted) >= rank:
            segment = segments_sorted[rank - 1]
            out[f"top{rank}_seg_class"] = segment["class_name"]
            out[f"top{rank}_density_group"] = segment["density_group"]
            out[f"top{rank}_area_ratio"] = float(segment["area"] / max(area_px, 1))
        else:
            out[f"top{rank}_seg_class"] = "unknown"
            out[f"top{rank}_density_group"] = "unknown"
            out[f"top{rank}_area_ratio"] = 0.0
    out["dominant_seg_class"] = out["top1_seg_class"]
    out["dominant_density_group"] = out["top1_density_group"]

    p05p95 = depth_versions["p05p95"]
    out.update(height_features(union_mask, p05p95, "union_p05p95"))
    out.update(height_features(union_mask, depth_versions["iqr_z"], "union_iqrz"))

    for segment in segments:
        group = str(segment["density_group"])
        mask = segment["mask"]
        out[f"area_group_{group}"] = float(out.get(f"area_group_{group}", 0.0) + int(segment["area"]))
        out[f"count_group_{group}"] = float(out.get(f"count_group_{group}", 0.0) + 1.0)
        hf = height_features(mask, p05p95, "tmp")
        out[f"pvol_group_{group}"] = float(out.get(f"pvol_group_{group}", 0.0) + hf.get("tmp_ring035_vol_plate_minus_food", 0.0) + hf.get("tmp_ring035_vol_food_minus_plate", 0.0))

    for group in density_groups:
        out[f"area_ratio_group_{group}"] = float(out.get(f"area_group_{group}", 0.0) / max(area_px, 1))

    return out


## 5. Feature extraction


In [ ]:
seg_model = YOLO(str(SEG_MODEL_PATH))
plate_model = YOLO(str(PLATE_MODEL_PATH))
cls_model = YOLO(str(CLS_MODEL_PATH)) if CLS_MODEL_PATH else None
depth_engine = DepthAnything3Module(model_id=DEPTH_MODEL_ID)

feature_rows: List[Dict[str, object]] = []
done_ids = set()
if RESUME_FEATURE_CACHE and FEATURES_CSV.exists():
    cached = pd.read_csv(FEATURES_CSV)
    feature_rows = cached.to_dict("records")
    done_ids = set(cached["dish_id"].astype(str))
    print(f"Loaded cached feature rows: {len(done_ids):,}")

pending = labels[~labels["dish_id"].astype(str).isin(done_ids)].copy()
print(f"Pending dishes: {len(pending):,}")


In [ ]:
errors = []
started_at = time.time()

for _, row in tqdm(list(pending.iterrows()), total=len(pending)):
    dish_id = str(row["dish_id"])
    image_path = Path(row["rgb_path"])
    try:
        image = Image.open(image_path).convert("RGB")
        width, height = image.size
        segments, seg_stats = predict_semantic_segments(seg_model, image)
        plate_mask, plate_stats = predict_plate_mask(plate_model, image)
        depth = np.asarray(depth_engine.get_depth_matrix(str(image_path)), dtype=np.float32)
        feature_rows.append({
            "dish_id": dish_id,
            "image_path": str(image_path.relative_to(PROJECT_ROOT)),
            "mass": float(row["mass"]),
            "calories": float(row["calories"]),
            "fat": float(row["fat"]),
            "carb": float(row["carb"]),
            "protein": float(row["protein"]),
            **seg_stats,
            **plate_stats,
            **compute_image_features(segments, plate_mask, depth, (height, width)),
            **predict_food101(cls_model, image),
        })
    except Exception as exc:
        errors.append({"dish_id": dish_id, "error": repr(exc)})

    if len(feature_rows) % CACHE_EVERY_N_ROWS == 0:
        pd.DataFrame(feature_rows).to_csv(FEATURES_CSV, index=False)

features = pd.DataFrame(feature_rows)
features.to_csv(FEATURES_CSV, index=False)
print(f"Feature rows: {len(features):,}")
print(f"Errors: {len(errors):,}")
print(f"Elapsed minutes: {(time.time() - started_at) / 60:.1f}")
features.head()


## 6. CatBoost training


In [ ]:
features = pd.read_csv(FEATURES_CSV)
features = features.replace([np.inf, -np.inf], np.nan).dropna(subset=["mass"])

report_cols = ["calories", "fat", "carb", "protein"]
drop_cols = ["dish_id", "image_path", "mass", *report_cols]
feature_cols = [c for c in features.columns if c not in drop_cols]
cat_cols = [
    c for c in feature_cols
    if pd.api.types.is_object_dtype(features[c])
    or pd.api.types.is_string_dtype(features[c])
    or isinstance(features[c].dtype, pd.CategoricalDtype)
]
num_cols = [c for c in feature_cols if c not in cat_cols]

features[cat_cols] = features[cat_cols].fillna("unknown").astype(str)
features[num_cols] = features[num_cols].fillna(0.0)

train_df, valid_df = train_test_split(features, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_valid = train_df[feature_cols], valid_df[feature_cols]
y_train = np.log1p(train_df["mass"].astype(float))
y_valid_log = np.log1p(valid_df["mass"].astype(float))
y_valid = valid_df["mass"].astype(float)

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
valid_pool = Pool(X_valid, y_valid_log, cat_features=cat_cols)
print(f"Train rows: {len(train_df):,}; valid rows: {len(valid_df):,}")
print(f"Features: {len(feature_cols)}; categorical: {len(cat_cols)}")


In [ ]:
model = CatBoostRegressor(**CATBOOST_PARAMS)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

pred_mass = np.clip(np.expm1(model.predict(valid_pool)), 0, None)
valid_report = valid_df[["dish_id", "mass", "calories", "fat", "carb", "protein"]].copy()
valid_report["pred_mass_g"] = pred_mass
valid_report["abs_error_g"] = np.abs(valid_report["mass"] - valid_report["pred_mass_g"])
valid_report["mass_bin"] = pd.cut(valid_report["mass"], bins=[0, 100, 250, 500, np.inf], labels=["0-100", "100-250", "250-500", "500+"])

for col in ["calories", "fat", "carb", "protein"]:
    per_g = valid_report[col].astype(float) / valid_report["mass"].astype(float).clip(lower=1e-6)
    valid_report[f"{col}_per_100g"] = per_g * 100.0
    valid_report[f"pred_{col}"] = per_g * valid_report["pred_mass_g"].astype(float)

metrics = {
    "valid_mae_g": float(mean_absolute_error(y_valid, pred_mass)),
    "valid_rmse_g": float(math.sqrt(mean_squared_error(y_valid, pred_mass))),
    "valid_r2": float(r2_score(y_valid, pred_mass)),
    "baseline_median_mae_g": float(mean_absolute_error(y_valid, np.full_like(y_valid, train_df["mass"].median(), dtype=float))),
    "bin_mae_g": {str(k): float(v) for k, v in valid_report.groupby("mass_bin", observed=False)["abs_error_g"].mean().to_dict().items()},
    "train_rows": int(len(train_df)),
    "valid_rows": int(len(valid_df)),
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "model_path": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    "features_csv": str(FEATURES_CSV.relative_to(PROJECT_ROOT)),
    "valid_report_csv": str(VALID_REPORT_CSV.relative_to(PROJECT_ROOT)),
}

model.save_model(MODEL_PATH)
valid_report.to_csv(VALID_REPORT_CSV, index=False)
METRICS_PATH.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metrics, indent=2, ensure_ascii=False))
valid_report[[
    "dish_id", "mass", "pred_mass_g", "abs_error_g",
    "pred_calories", "pred_fat", "pred_carb", "pred_protein",
    "calories_per_100g", "fat_per_100g", "carb_per_100g", "protein_per_100g",
]].head(20)
